# Session 6 Lab — An Agent You Can Audit, and the Retrieval It Reads From

Five blocks: the agent loop first, then the retrieval it reads from, and governance last. The last column
names the deck that introduces each block. Decks 6.1 and 6.2 each end on a "Your turn" frame that names its
blocks. Each block ends with a **gate**: a bound fixed before the run.
Nothing here needs a GPU or an account. One cell, B1's Step 2, fetches the toy's encoder the first time it runs
(about 90 MB, after `pip install sentence-transformers`); without them it reads the deck's own matrix instead,
and from a clean kernel the whole notebook runs offline in a few minutes.

**One bounded question, all the way through:**

> *What did the Committee say about inflation at a given meeting, and what did the ten-year Treasury
> yield do over the following month — cite the statement.*

It is bounded in three ways that matter. It names a **document set** (245 FOMC statements), it names a
**number** that lives outside those documents (a yield panel), and it demands a **citation**, so an
answer that cannot point at a sentence has failed whether or not it is true.

| Block | min | You will | Deck |
|---|---|---|---|
| **A** | 20 | run a scripted model through real tools and read the loop line by line, in Deck 6.1's shape | 6.1 |
| **B** | 35 | build Deck 6.2's toy, five steps, and keep it as one file for your own folder; then chunk the corpus two ways and build TF-IDF, BM25 and reciprocal rank fusion in numpy | 6.2 |
| **C** | 25 | grade retrieval on 26 questions, fit a refusal threshold, break the pipeline three ways, and hand the passage to a reader | 6.2 |
| **D** | 15 | break an answer into atomic claims and mark each one against the retrieved text, by hand | 6.2 |
| **E** | 15 | write budget, permission and verifier as code, and price reliability | 6.1 |

**What a gate is here.** Every gate is on **procedure** or on **reproduction**, never on an outcome.
A procedure gate checks how the run was done: hashes checked before a byte is read, gold anchors proved to
name statements the corpus holds, the two reader conditions putting the answer sentence at identical window
positions, every tool call passing an allow-list and a gate before it runs, and the deck's toy matrix
checked against its manifest before a row of it is read. A reproduction gate checks that a digit some
frozen artefact already carries comes back.

**Two results in this lab go against the grain, and neither is gated in a direction.**
Reciprocal rank fusion does **not** beat BM25 here — mean reciprocal rank 0.933 against 0.958. The
256-character reader, handed the exact passage retrieval found, predicts the answer sentence about
**three per cent** more cheaply than it does from a passage ten years away. Both are results to teach.
No gate in this notebook requires fusion to beat BM25, and none requires the reader to succeed.

The two modules behind the blocks live in `session06_assets/`: `session06_agent.py` (A, E) and
`session06_rag.py` (B, C, D). They do not import each other — the agent's own `fomc_search` tool carries a
deliberately plain retriever, because the subject of A is the loop, not the ranking. The numbers the
slides quote come from full-budget runs of exactly these functions, stored in `recorded_agent.json` and
`recorded_rag.json`. A third file, `toy_rag.py`, is B1's toy as one file to take away; nothing here imports it
but B1's last cell.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")       # set before numpy is imported; shared machines need it
%matplotlib inline
import collections, csv, itertools, json, re, sys, textwrap, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LAB = Path.cwd() if (Path.cwd() / "session06_assets").is_dir() else Path.cwd() / "labs"
ASSETS = LAB / "session06_assets"
sys.path.insert(0, str(ASSETS))
import session06_rag as rag            # blocks B, C, D
import session06_agent as ag           # blocks A, E

GATE = json.loads((ASSETS / "lab_gates.json").read_text(encoding="utf-8"))
REC = {k: json.loads((ASSETS / f"recorded_{k}.json").read_text(encoding="utf-8"))
       for k in ("rag", "agent")}
TEAL, RED, MUTED = "#006D77", "#A3473B", "#5C686B"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
T_START = time.time()
GATES = []

def gate(name, ok, detail=""):
    GATES.append((name, bool(ok)))
    print(f"GATE {name}: {'pass' if ok else 'FAIL'}   {detail}")
    assert ok, f"gate {name} failed: {detail}"

print("numpy", np.__version__, "| pandas", pd.__version__, "| torch", rag.torch.__version__)
print("recorded runs:", ", ".join(f"recorded_{k}.json" for k in REC))

# Block A — The loop, offline

An agent is not a model. It is a **loop**: send the conversation, get back either text or a request to call a
tool, run the tool, append the result, send the conversation again. That is the whole idea, and everything
that makes agents expensive, slow, or dangerous follows from those five lines.

This is the loop of Deck 6.1's Part C, run: your instruction, the model's tool call in JSON, the harness's
observation, and rules that can refuse a call. A2 prints the run in the slide's shape.

Nothing in this block reaches the network. The model is a **fixed script** of seven turns; the tools are
real code over real files — the FOMC corpus and Session 2's yield panel. Substituting one stubbed function
(`model.create`) for a real API call is the only difference between this loop and a production one, and
every number below is computed by running it.

## A1. Tools are code plus a schema

In [ ]:
tools = ag.build_tools()
print(f"{len(tools)} tools, each a function plus a JSON schema the model is shown\n")
for t in tools:
    params = ", ".join(t.input_schema["properties"])
    required = ", ".join(t.input_schema.get("required", []))
    print(f"{t.name:<18}({params})")
    print(f"{'':<18}required: {required or 'none'}")
    print(f"{'':<18}{textwrap.shorten(t.description, 82)}")
print(f"\nthe allow-list for this task: {sorted(ag.ALLOWED)}")
print("A tool the loop can describe but not call is the point of the list. The model chooses arguments;")
print("the allow-list chooses whether the call happens at all.")

## A2. Seven turns, traced

Turn 4 asks for a shell command, which is not on the allow-list. Turn 5 writes a CSV with no source column,
which the gate stops. Both are in the script on purpose, so the refusal and the gate are **exercised by the
loop** rather than described next to it.

In [ ]:
run = ag.scripted_run(trace=True)
print(f"\nstopped: {run['stop']} after {run['turns']} turns; "
      f"{run['tokens_sent_total']:,} tokens sent, {run['tokens_out_total']:,} out")
print(f"files written: {run['files_written']}")

In [ ]:
print("The same run, in the shape of Deck 6.1's 'How the Agent Runs a Step':\n")
for who, what in ag.trace_rows(run):
    print(f"{who:<11}{what}")
print("\nYou write the first line. The model writes each JSON line; the harness runs it and hands back what")
print("happened. Two calls never ran: the allow-list refused one, and a gate refused the other.")

## A3. The transcript, read line by line

The message list is the agent's entire memory. There is nothing else: no hidden state, no variables carried
between turns, nothing the loop knows that is not in this list (Deck 6.1: "The Session Is a List, Not a
Connection"). It is the run above as the API carries it: each tool call a `tool_use` block with an id, a name
and an input; each observation a `tool_result` block inside a user message. Read it.

In [ ]:
for i, message in enumerate(run["messages"]):
    content = message["content"]
    if isinstance(content, str):
        print(f"[{i}] {message['role']:<10} text        {textwrap.shorten(content, 88)}")
        continue
    for block in content:
        kind = block.get("type")
        if kind == "text":
            print(f"[{i}] {message['role']:<10} text        {textwrap.shorten(block['text'], 88)}")
        elif kind == "tool_use":
            print(f"[{i}] {message['role']:<10} tool_use    {block['name']}"
                  f"({json.dumps(block['input'])[:62]})")
        elif kind == "tool_result":
            flag = "  [error]" if block.get("is_error") else ""
            print(f"[{i}] {message['role']:<10} tool_result {textwrap.shorten(str(block['content']), 76)}"
                  f"{flag}")
print(f"\n{len(run['messages'])} messages, and the final answer is not among them — the loop breaks on")
print("the turn that stops asking for tools, so the text below is what the seventh reply carried:\n")
print(textwrap.fill(run["final_text"], 100, initial_indent="   ", subsequent_indent="   "))
print("\nA tool RESULT is a user message. The loop plays the part of the user when it hands back what a")
print("tool returned, which is why an error from a tool reads to the model exactly like a person saying")
print("something went wrong.")

recorded_run = REC["agent"]["replayed_run"]
ok = (run["turns"] == GATE["agent_turns"] == recorded_run["turns"]
      and run["stop"] == GATE["agent_stop"]
      and run["tokens_sent_total"] == GATE["agent_tokens_sent_total"]
      and run["tokens_out_total"] == GATE["agent_tokens_out_total"]
      and run["ledger"] == recorded_run["ledger"]
      and run["files_written"] == GATE["agent_files_written"]
      and run["verification"]["counts"] == recorded_run["verification"]["counts"]
      and run["verification"]["n_checks"] == GATE["agent_verifier_checks"])
gate("A1", ok, f"{run['turns']} turns, {run['tokens_sent_total']:,} tokens sent and the whole ledger "
               f"reproduce the recorded run; the verifier's {run['verification']['n_checks']} checks come "
               f"back as recorded")

## A4. The whole list is re-sent every turn

Turn $k$ sends a list of size $k$. If each turn adds a fixed amount of new text, the running total of tokens
sent grows with the **square** of the turn count while the new text grows linearly. This is the single most
expensive fact about the agentic loop, and it is arithmetic, not a vendor's pricing decision. Deck 6.1 drew it
as "What Crosses the Wire": the whole list, re-sent every turn.

In [ ]:
ledger = pd.DataFrame(run["ledger"])
print(ledger.to_string(index=False))
print(f"\nnew text across all {run['turns']} turns: {ledger['tokens_new'].sum():,} tokens")
print(f"actually sent:                 {ledger['tokens_sent'].sum():,} tokens "
      f"— {ledger['tokens_sent'].sum() / max(ledger['tokens_new'].sum(), 1):.1f} times as much")

curve = ag.cost_curve()
model_curve = ag.resend_cost(12, curve["new_tokens_per_turn"])
fig, ax = plt.subplots(figsize=(8.0, 3.2))
ax.plot(range(1, 13), model_curve, marker="o", markersize=4, color=MUTED, linewidth=1.4,
        label=f"{curve['new_tokens_per_turn']} new tokens a turn, list re-sent each time")
ax.plot(ledger["turn"], ledger["cumulative_sent"], marker="s", markersize=4, color=TEAL, linewidth=1.6,
        label="this run, measured")
ax.set_xlabel("turn"); ax.set_ylabel("cumulative tokens sent")
ax.legend(frameon=False, fontsize=8); ax.grid(alpha=0.25)
ax.set_title("Linear new text, quadratic bill", fontsize=11)
fig.tight_layout(); plt.show()

sent_matches = all(row["tokens_sent"] >= row["tokens_new"] > 0 for row in run["ledger"])
monotone = all(a["cumulative_sent"] < b["cumulative_sent"]
               for a, b in zip(run["ledger"], run["ledger"][1:]))
accounting = all(row["tokens_sent"] == sum(r["tokens_new"] for r in run["ledger"][:i + 1])
                 for i, row in enumerate(run["ledger"]))
ok = sent_matches and monotone and accounting
gate("A2", ok, "every turn's tokens_sent equals the sum of all new text up to that turn, and the "
               "cumulative total is strictly increasing — the ledger accounts for the re-send exactly")

## A5. What the loop refused to do

Two calls in this run never reached a tool. One was stopped by the **allow-list**, before the tool ran at
all; the other by a **PreToolUse hook** that read the pending call's arguments. They are different mechanisms
with different failure modes, and Block E takes the second one apart.

A **hook** is a command the harness runs at a fixed moment, here just before a tool call, with the pending call
on its standard input. If it exits with code 2 the call is refused, and what it printed goes back to the model
as the observation. Hooks are set in `settings.json`, the file Deck 6.1's file tree labels "your rules and
hooks".

In [ ]:
for b in run["blocked"]:
    print(f"{b['tool']:<8} stopped by {b['by']}")
    print(textwrap.fill(f"   {b['reason']}", 100, subsequent_indent="   "))
print()

demo = ag.budget_demo(max_steps=GATE["budget_demo_max_steps"])
print(f"the same run under max_steps = {demo['max_steps']}: {demo['turns']} turns, stop = {demo['stop']}, "
      f"{demo['tokens_sent_total']:,} tokens sent")
print(f"final answer: {demo['final_text'] or '(none — it stopped mid-task)'}")
print(f"\n{demo['note']}. A budget that binds does not produce a shorter answer. It produces no answer,")
print("and the tokens are spent either way.")

ok = (len(run["blocked"]) == GATE["agent_blocked"]
      and sorted({b["by"] for b in run["blocked"]}) == sorted(GATE["agent_blocked_by"])
      and sorted(ag.ALLOWED) == sorted(GATE["agent_allow_list"])
      and demo["turns"] == GATE["budget_demo_turns"] and demo["stop"] == GATE["budget_demo_stop"]
      and demo["final_text"] == "" and demo["tokens_sent_total"] == GATE["budget_demo_tokens_sent_total"])
gate("A3", ok, f"{len(run['blocked'])} calls stopped before their tools ran, one by each mechanism; the "
               f"budget demonstration halts at {demo['turns']} turns with no answer, as recorded")

# Block B — Build the index

Deck 6.2 built a library in five steps. B1 runs them here, for real: it cuts the 245 statements into passages,
encodes them, keeps the matrix, asks the slide's question, writes the prompt, and checks the toy on graded
questions. It finds what the slide found, the right topic and the wrong year, and it ends with the same steps as
one file you can point at your own folder. B2–B6 then build what the toy does not have (chunks that carry their
date, a keyword score, fusion) and grade every choice rather than assert it. The `fomc_search` tool the loop
called in Block A was a deliberately plain retriever, handed over without explanation so that the subject stayed
the loop; B2–B6 open it.

## B1. Deck 6.2's toy, built

The listings below are Deck 6.2's Steps 1–4, run on the same 245 statements; Step 5 then checks them.

**One install, one download.** Step 2 needs `sentence-transformers` (`pip install sentence-transformers`; NumPy
and PyTorch are already in the course environment), and its first run fetches the encoder, all-MiniLM-L6-v2:
about 90 MB, read from disk after that. In mainland China, set the environment variable
`HF_ENDPOINT=https://hf-mirror.com` before you start Jupyter. Without the package or the network, Step 2 says so
and reads the matrix the deck's own run wrote on 20 October 2026 instead, shipped in `toy_index/` and checked
against its manifest before a row is read; every other cell runs unchanged.

In [ ]:
# Step 1, as on the slide. The folder is the lab's corpus, and the encoding is named: on a Chinese-locale
# Windows laptop Python would otherwise read these files as GBK, and 54 of them carry a curly quote or a dash.
docs, ids = [], []   # passage, address
for f in sorted((ASSETS / "fomc").glob("*.txt")):
    paras = f.read_text("utf-8").split("\n\n")
    for n, p in enumerate(paras):
        if len(p) > 40 and p[0] != "#":
            docs.append(p)
            ids.append(f"{f.stem}#{n}")
n_statements = len({a.split("#")[0] for a in ids})
print(f"{len(docs):,} passages from {n_statements} statements. The first:")
print(f"   [{ids[0]}] {docs[0][:88]}...")

In [ ]:
# Step 2, as on the slide: encode every passage once and keep the matrix. The revision pins the weights the deck
# used ("pin the file, not the name"); the matrix and its addresses go to a folder of yours, in one order.
toy = rag.load_toy_index()     # the deck's own run, every file's sha256 checked first: the check, and the fallback
MY_INDEX = LAB / "my_toy_index"
try:
    import sentence_transformers as st
    name = "sentence-transformers/all-MiniLM-L6-v2"
    enc = st.SentenceTransformer(name, revision=toy["manifest"]["revision"])
    t0 = time.time()
    V = enc.encode(docs, normalize_embeddings=True)
    built_here = True
    print(f"encoded {len(docs):,} passages in {time.time() - t0:.0f} s")
except Exception as err:       # no package, or no network for the first download
    enc, V, built_here = None, toy["V"], False
    print(f"No encoder here ({type(err).__name__}). Run  pip install sentence-transformers  and rerun this cell;")
    print("until then the matrix the deck's own run wrote stands in, and every later cell runs as it would.")
# V: N x 384, one row per passage, length one
MY_INDEX.mkdir(exist_ok=True)
np.save(MY_INDEX / "vectors.npy", V)
with open(MY_INDEX / "ids.json", "w", encoding="utf-8") as fh:
    json.dump(ids, fh)
lengths = np.linalg.norm(V, axis=1)
gap = float(np.abs(V - toy["V"]).max())
print(f"V: {V.shape[0]:,} x {V.shape[1]}, one row per passage, each of length {lengths.min():.4f} to {lengths.max():.4f},")
print(f"saved with its addresses in {MY_INDEX.name}/. {'Built here' if built_here else 'Read from the deck'}: "
      f"the largest difference from the deck's matrix is {gap:.1e}.")

In [ ]:
# Step 3, as on the slide: the question goes through the same encoder, and one product scores every row.
question = toy["question_text"]["toy"]
q = enc.encode([question], normalize_embeddings=True)[0] if built_here else toy["questions"]["toy"]
s = V @ q                 # cosine, all rows
top = np.argsort(-s)[:5]  # the five nearest
print(f"question: {question}\n")
for i in top:
    print(ids[i], s[i].round(2), docs[i][:60])

The slide's table, digit for digit: the right topic and the wrong year, five times over. The encoder reads
meaning, and the date is not in the meaning; it is in the address. Deck 6.2's fix is a filter on the address,
and it takes one line.

In [ ]:
keep = np.array([a.startswith("fomc_202203") for a in ids])   # the address carries the date
march = [i for i in np.argsort(-s) if keep[i]]
print(f"{len(march)} passages pass the filter, all from the statement of 16 March 2022, best first:\n")
for i in march:
    print(ids[i], s[i].round(2), docs[i][:60])
order = list(np.argsort(-s))
best_rank = order.index(march[0]) + 1
elevated_rank = order.index(ids.index("fomc_20220316#1")) + 1     # the paragraph that says it
print(f"\nWithout the filter the best of them ranks {best_rank:,} of {len(ids):,}; the paragraph "
      f"that says 'Inflation remains elevated' ranks {elevated_rank:,}.")

The filter finds the meeting. Now read what comes first: the paragraph in which the Committee restates its
2 percent objective and raises the rate. The paragraph that actually says *"Inflation remains elevated"*
comes fourth of five, because most of its sentences are about jobs. The encoder pools a whole passage into
one point, and a passage about two things lands between them. A filter fixes
which statement; it cannot fix what a point means. B3 cuts the statements differently for that reason.

The same listings run with any encoder that turns a text into a row of length one. Here is one with no
download at all, TF-IDF in a dozen lines of numpy. `V @ q` is still a cosine.

In [ ]:
words = lambda text: re.findall(r"[a-z0-9']+", text.lower())
vocab = {w: k for k, w in enumerate(sorted({w for d in docs for w in words(d)}))}

def counts(texts):
    M = np.zeros((len(texts), len(vocab)), dtype=np.float32)
    for r, text in enumerate(texts):
        for w, c in collections.Counter(words(text)).items():
            if w in vocab:
                M[r, vocab[w]] = 1 + np.log(c)       # a word said six times is not six times as much
    return M

C = counts(docs)
idf = np.log((len(docs) + 1) / ((C > 0).sum(axis=0) + 1)) + 1
Vc = C * idf
Vc /= np.linalg.norm(Vc, axis=1, keepdims=True)      # rows of length one, as in Step 2
qc = counts([question])[0] * idf
qc /= np.linalg.norm(qc)
sc = Vc @ qc
for i in np.argsort(-sc)[:5]:
    print(ids[i], sc[i].round(2), docs[i][:60])
print(f"\n{len(vocab):,} words in the vocabulary; '2022' is one of them: {'2022' in vocab}.")
print("No passage contains the year, so no encoder of the text can see it. Only the address can.")

In [ ]:
# Step 4, as on the slide: the prompt, built from Step 3's five rows and printed, not sent.
context = "\n\n".join(f"[{ids[i]}] {docs[i]}"
                      for i in top)
prompt = f"""Answer only from these passages.
Tag every claim with its [address].
If the answer is not in them, say so.

{context}

Question: {question}"""
print(prompt[:620] + "\n...\n")
print(f"{len(prompt):,} characters. Nothing is sent from here: paste the prompt into a chat window, hand it to your")
print("agent, or pass it to any model's API. Given these five passages the right answer is a refusal; given the")
print("March 2022 passages, an answer with their addresses.")

deck = toy["manifest"]["deck_run"]
d = GATE["toy_score_decimals"]
rows = sorted([ids[i], round(float(s[i]), d)] for i in top)      # as a set: rows 4 and 5 tie exactly
ok = (len(docs) == deck["passages"] == GATE["toy_passages"]
      and toy["ids"] == ids and V.shape == (GATE["toy_passages"], GATE["toy_dimension"])
      and toy["hashes_checked"] == GATE["toy_files"]
      and (gap <= GATE["toy_build_tolerance"] if built_here else gap == 0.0)
      and question == deck["question"]
      and rows == sorted([a, round(x, d)] for a, x in deck["top5"])
      and len(prompt) == deck["prompt_chars"])
gate("B1", ok, f"Deck 6.2's toy {'built here' if built_here else 'read from the deck'} and checked against the deck's "
               f"run: {len(docs):,} passages at its addresses, {toy['hashes_checked']} files hash-checked, the matrix "
               f"within {gap:.1e}, the slide's five rows to {d} decimals, and a {len(prompt):,}-character prompt")

### B1, Step 5: check the retrieval end on graded questions

Deck 6.2's Step 5 checks both ends: does the right passage come back, and is every claim in one? The second end
needs an answer, and Block D marks one by hand. The first end can be measured now. B2 introduces the lab's 26
graded questions properly (20 the corpus answers, each with its anchor statement, and 6 it cannot); here they only
go through the toy. Most of them name a meeting, so each can also be asked with that date as the filter on the
address that rescued the slide's question.

In [ ]:
# Step 5: is the right statement among the five nearest passages? Every graded question, with and without the
# date it names as a filter on the address. The questions go through Step 2's encoder (offline, the deck's run).
graded = rag.load_gold()
MONTHS = "january february march april may june july august september october november december".split()

def named_date(text):
    "The date a question names, written as the address writes it (19940204), or its year; None if neither."
    m = re.search(r"(%s) (\d{1,2}), ((?:19|20)\d\d)" % "|".join(MONTHS), text.lower())
    if m:
        return f"{m.group(3)}{MONTHS.index(m.group(1)) + 1:02d}{int(m.group(2)):02d}"
    m = re.search(r"\b(?:19|20)\d\d\b", text)
    return m.group(0) if m else None

Qg = (enc.encode([g.question for g in graded], normalize_embeddings=True) if built_here
      else np.stack([toy["questions"][g.id] for g in graded]))
statement = [a.split("#")[0][len("fomc_"):] for a in ids]     # fomc_20220316#1 -> 20220316
checks = []
for g, qg in zip(graded, Qg):
    order = np.argsort(-(V @ qg))
    date = named_date(g.question)
    kept = [i for i in order if date is not None and statement[i].startswith(date)]
    anchors = {a.replace("-", "") for a in g.anchors}
    checks.append(dict(id=g.id, answerable=g.answerable, date=date, left=len(kept),
                       plain=any(statement[i] in anchors for i in order[:5]),
                       dated=any(statement[i] in anchors for i in kept[:5])))
answerable = [c for c in checks if c["answerable"]]
unanswerable = [c for c in checks if not c["answerable"]]
plain = sum(c["plain"] for c in answerable)
dated = sum(c["dated"] for c in answerable)
emptied = [c["id"] for c in unanswerable if c["date"] is not None and c["left"] == 0]
print(f"{len(answerable)} answerable questions: the right statement among the five nearest for {plain} of them "
      f"without the filter, for {dated} with it")
print(f"   missed without the filter: {', '.join(c['id'] for c in answerable if not c['plain'])}")
print(f"{len(unanswerable)} unanswerable: the date filter leaves no passage at all for {len(emptied)} "
      f"({', '.join(emptied)}), a refusal before any model is asked")
print(f"   left for a threshold to catch (Block C): {', '.join(c['id'] for c in unanswerable if c['id'] not in emptied)}")

Without the filter the toy finds the right statement for fewer than half of the questions it should answer: the
encoder reads the topic, and three decades of statements share their topics. With the date each question names, it
finds all of them. Four of the six it cannot answer name a meeting the corpus does not hold, and the filter leaves
nothing to answer from. The other two need a threshold on the evidence; Block C fits one.

### B1, take it home: the same steps as one file

`session06_assets/toy_rag.py` holds Steps 1–4 as seven short functions and a command line. Copy it into a project
of your own, point it at a folder of plain-text files saved as UTF-8, and the toy reads your library:

    python toy_rag.py build my_notes/ my_index/
    python toy_rag.py ask my_index/ "what did I conclude about the discount factor?"
    python toy_rag.py ask my_index/ "the same question" --where 2019     # only files whose name starts with 2019

Each of three things changes in one place: the folder (`parse`); the encoder (`load_encoder`: any
sentence-transformers model, a multilingual one if your library is not in English); and what answers (send
`make_prompt`'s text to any chat model). Step 5 is yours to write: a dozen questions with known answers, and a few
your library cannot answer, written before you look at a result. Deck 6.2's skill frame shows an agent running the
same file as a tool.

In [ ]:
# The same steps through toy_rag.py, on the lab's folder. For your own library, point FOLDER at a folder of .txt files.
import toy_rag
FOLDER = ASSETS / "fomc"
docs2, ids2 = toy_rag.parse(FOLDER)
if built_here:      # enc is Step 2's encoder; on its own, toy_rag.load_encoder() makes one
    toy_rag.save(MY_INDEX, toy_rag.encode(enc, docs2), ids2, docs2)
    V2, ids2, docs2, meta = toy_rag.load(MY_INDEX)
    q2 = toy_rag.encode(enc, [question])[0]
else:
    V2, q2 = toy["V"], toy["questions"]["toy"]
hits = toy_rag.search(V2, q2, k=5)
for i, score in hits:
    print(f"{ids2[i]}  {score:.2f}  {docs2[i][:60]}")
prompt2 = toy_rag.make_prompt(question, hits, ids2, docs2)
march2 = toy_rag.search(V2, q2, k=5, keep=np.array([a.startswith("fomc_202203") for a in ids2]))
same_rows = sorted((ids2[i], round(x, 2)) for i, x in hits) == sorted((ids[i], round(float(s[i]), 2)) for i in top)
print(f"\nStep 3's five rows again: {same_rows}. Step 4's prompt again: {prompt2 == prompt}. "
      f"With --where fomc_202203: {len(march2)} passages, all from that meeting.")

ok = (ids2 == ids and same_rows and prompt2 == prompt
      and len(answerable) == GATE["gold_in_corpus"] and len(unanswerable) == GATE["gold_out_of_corpus"]
      and plain == GATE["toy_hit5_plain"] and dated == GATE["toy_hit5_dated"]
      and len(emptied) == GATE["toy_filter_empties"])
gate("B1b", ok, f"Step 5 reproduced ({plain} and {dated} of {len(answerable)} answerable questions; the filter "
                f"empties {len(emptied)} of {len(unanswerable)} unanswerable), and toy_rag.py returns the toy's rows and prompt")

## B2. The corpus, and the questions it will be graded on

The statements are the same 245 Session 5 used, hash-checked against the same manifest before a byte is read.
The graded question set is new: 26 questions, **20 the corpus answers and 6 it cannot**. The six are not
absurd — two ask about a real central bank this corpus mentions but never speaks for, two about meetings
outside the window, two about subjects a monetary policy statement has never covered. Block D needs them:
a refusal threshold fitted with only answerable questions in the set collapses to zero.

Before anything else, every anchor date in the gold set is checked against the corpus. An anchor with a typo
scores as a permanent miss, recall falls, and nothing looks wrong.

In [ ]:
statements = rag.load_statements()                 # every file's sha256 checked against the manifest
gold = rag.load_gold()
check = rag.check_gold_against_corpus(gold, statements)      # raises if an anchor names a missing statement
reflow = rag.reflow_report(statements)

print(f"corpus        {len(statements)} statements, {statements[0].date} to {statements[-1].date}")
print(f"              {reflow['paragraphs']:,} paragraphs; {reflow['statements_repaired']} statements "
      f"arrived hard-wrapped and were reflowed before anything read them")
print(f"gold set      {check['questions']} questions: {check['in_corpus']} the corpus answers, "
      f"{check['out_of_corpus']} it cannot; {len(check['anchors'])} distinct anchor statements, all present")
print()
print("the bounded task, on one meeting:")
print(f"   {ag.PROMPT}")

ok = (len(statements) == GATE["corpus_statements"]
      and statements[0].date == GATE["corpus_first"] and statements[-1].date == GATE["corpus_last"]
      and reflow["paragraphs"] == GATE["corpus_paragraphs"]
      and check["questions"] == GATE["gold_questions"]
      and check["in_corpus"] == GATE["gold_in_corpus"]
      and check["out_of_corpus"] == GATE["gold_out_of_corpus"]
      and len(check["anchors"]) == GATE["gold_anchors"]
      and check["anchors_checked"] is GATE["corpus_hashes_checked"])
gate("B2", ok, f"{len(statements)} statements, every hash matched before it was read; "
               f"{check['questions']} graded questions and all {len(check['anchors'])} anchors present")

## B3. Two ways to cut a document up

**Fixed-size** chunking counts 500 characters and stops. It is the default in every tutorial, it is trivial
to implement, and it knows nothing about the document.

**Document-aware** chunking respects the statement's own structure: paragraphs are classified into sections
(decision, assessment, guidance, implementation, objectives, vote, other), runs of same-section paragraphs
are merged up to 800 characters, and a split — when one is needed — falls at a sentence end. Every chunk
carries its date and its section, and is indexed with them, which is what lets a *dated* question be answered.

800 characters is not a taste. `rag.chunk_size_sweep` grades the whole pipeline at 600, 800, 1000 and 1400
and 800 wins every column; the sweep is recorded in `recorded_rag.json` under `chunking.size_sweep`.

In [ ]:
doc_chunks = rag.document_chunks(statements)                 # 800 characters, section-aware, sentence-safe
fixed_chunks = rag.fixed_size_chunks(statements)             # 500 characters, no overlap, no idea where it is

reports = {"document": rag.chunk_report(doc_chunks), "fixed": rag.chunk_report(fixed_chunks)}
print(f"{'':<12}{'chunks':>8}{'median chars':>14}{'opens mid-sentence':>21}{'closes mid-sentence':>21}")
for name, r in reports.items():
    print(f"{name:<12}{r['n_chunks']:>8,}{r['characters']['median']:>14.0f}"
          f"{r['opens_mid_sentence']:>21,}{r['closes_mid_sentence']:>21,}")
print(f"\ndocument-aware sections: " +
      "  ".join(f"{k} {v}" for k, v in reports["document"]["sections"].items()))
print(f"fixed-size sections:     {list(reports['fixed']['sections'])[0]} — it has none, because it never "
      f"looked at the document")

sweep = pd.DataFrame([{"max_chars": r["max_chars"], "chunks": r["chunks"],
                       "median": r["median_characters"],
                       "bm25 r1": r["recall_at_1"]["bm25"],
                       "bm25 MRR": r["mean_reciprocal_rank"]["bm25"]}
                      for r in REC["rag"]["chunking"]["size_sweep"]])
print(f"\nwhy 800, from the recorded sweep:\n{sweep.to_string(index=False)}")

ok = (reports["document"]["n_chunks"] == GATE["document_chunks"]
      and reports["fixed"]["n_chunks"] == GATE["fixed_chunks"]
      and reports["document"]["opens_mid_sentence"] == GATE["document_opens_mid_sentence"]
      and reports["document"]["closes_mid_sentence"] == GATE["document_closes_mid_sentence"]
      and reports["fixed"]["opens_mid_sentence"] == GATE["fixed_opens_mid_sentence"]
      and reports["fixed"]["closes_mid_sentence"] == GATE["fixed_closes_mid_sentence"]
      and len(reports["document"]["sections"]) == GATE["document_sections"])
gate("B3", ok, f"{reports['document']['n_chunks']:,} document-aware chunks, none opening or closing "
               f"mid-sentence; {reports['fixed']['n_chunks']:,} fixed-size chunks, "
               f"{reports['fixed']['opens_mid_sentence']:,} of them cut into a sentence")

### What a boundary costs, in sentences

A chunk boundary that falls inside a sentence leaves two halves, and **neither half is the sentence**. One
chunk can be retrieved and quoted; the sentence cannot. Count them.

In [ ]:
damage = rag.boundary_damage(fixed_chunks, statements)
print(f"{damage['sentences_cut']:,} of {damage['sentences']:,} sentences "
      f"({100 * damage['share_cut']:.1f}%) are cut by a fixed-size boundary.\n")
example = damage["examples"][0]
print(f"one of them, from {example['date']}:")
print(textwrap.fill(f"   first half   ...{example['first_half'][-70:]}", 100))
print(textwrap.fill(f"   second half  {example['second_half'][:70]}...", 100))
print("\nEither chunk on its own reads as a complete thought and is not one.")

## B4. Two scores, both written out in numpy

A query and a chunk both become vectors over the same vocabulary, fitted on the chunks themselves.

**TF-IDF with cosine similarity.** $\mathrm{tf} = 1 + \log(\text{count})$ for a term that occurs and zero
otherwise — sublinear, because a paragraph that says *inflation* six times is not six times as much about
inflation. $\mathrm{idf} = \log\frac{N+1}{\mathrm{df}+1} + 1$. Rows are L2-normalised, so a score is a cosine
in $[0, 1]$ and **scores from different queries are on the same scale**. Hold on to that: it is the only
reason a single refusal threshold can exist at all.

**BM25**, in the Robertson–Sparck Jones form:

$$\mathrm{score}(q,d) = \sum_{t \in q} \mathrm{idf}(t)\,\frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\left(1 - b + b\frac{|d|}{\mathrm{avgdl}}\right)}, \qquad \mathrm{idf}(t) = \ln\!\left(1 + \frac{N - \mathrm{df}(t) + 0.5}{\mathrm{df}(t) + 0.5}\right)$$

$k_1$ controls how fast term frequency saturates; $b$ controls length normalisation. The defaults are
$k_1 = 1.5$, $b = 0.75$. A BM25 score has **no ceiling and is not comparable across queries** — which is
exactly why the fusion below reads ranks and not scores.

In [ ]:
doc_retriever = rag.FusionRetriever(doc_chunks)        # fits the term space once; both indexes share it
fixed_retriever = rag.FusionRetriever(fixed_chunks)
space = doc_retriever.space
print(f"term space fitted on {space.n_documents:,} chunks: {space.n_terms:,} distinct terms")
print(f"BM25 settings k1 = {doc_retriever.bm25.k1}, b = {doc_retriever.bm25.b}; "
      f"average chunk length {doc_retriever.bm25.average_length:.1f} tokens\n")

question = "what did the Committee say about inflation at the meeting of March 16, 2022"
print(f"query: {question}\n")
for name in ("tf-idf", "bm25"):
    index = doc_retriever.named()[name]
    print(f"{name}")
    for rank, (i, score) in enumerate(index.search(question, top_k=3), start=1):
        c = doc_chunks[i]
        print(f"   {rank}. {score:>8.4f}   {c.date}  {c.section:<14}{c.text[:56]}...")
print("\nThe cosine is bounded and the BM25 score is not. Two numbers, two scales, one ordering problem.")

## B5. Reciprocal rank fusion: ranks only, on purpose

You cannot add a cosine in $[0,1]$ to an unbounded BM25 score. You can add their **ranks**:

$$\mathrm{RRF}(d) = \sum_{\text{rankings}} \frac{1}{k + \mathrm{rank}(d)}, \qquad k = 60$$

Cormack, Clarke and Büttcher (SIGIR 2009). The arithmetic needs no training and no score calibration, and
$k = 60$ does one job: it flattens the top of the curve, so the gap between rank 1 and rank 2 is
$\frac{1}{61} - \frac{1}{62} = 0.00026$ and a document has to do well in **more than one** ranking to rise.

That flattening is the whole method, and it is also its cost. Watch what it does to a document that one
retriever is sure about.

In [ ]:
# Four documents, two rankings. Retriever 1 is confident and right; retriever 2 is confident and wrong.
rankings = [[0, 1, 2, 3], [1, 2, 3, 0]]
fused = rag.reciprocal_rank_fusion(rankings, n_documents=4, k=rag.RRF_K)
print(f"k = {rag.RRF_K}\n")
print(f"{'doc':<6}{'rank in A':>11}{'1/(k+r)':>11}{'rank in B':>11}{'1/(k+r)':>11}{'fused':>10}{'':>4}")
for d in range(4):
    ra, rb = rankings[0].index(d) + 1, rankings[1].index(d) + 1
    print(f"{d:<6}{ra:>11}{1 / (rag.RRF_K + ra):>11.5f}{rb:>11}{1 / (rag.RRF_K + rb):>11.5f}"
          f"{fused[d]:>10.5f}")
print(f"\nfused order: {list(rag.rank_of(fused))}")
print("Document 0 was first in A and last in B, and fusion puts it third. The arithmetic cannot see that A")
print("put it first by a mile and B put it last by a whisker, because it threw the scores away.")

## B6. The grading table: six columns, one corpus

recallk is the share of answerable questions with a chunk from the right statement somewhere in the top
$k$ — recall and not precision, because a question here has one right document and the retriever is allowed
several guesses. **MRR** is the mean of $1/\mathrm{rank}$ of the first correct document; it is the recallk
curve in one number, and it is harsh in the right way, since moving the answer from rank 3 to rank 1 is worth
more than moving it from rank 30 to rank 10.

Three retrievers, two chunkings, the same 20 questions.

In [ ]:
grades = {}
for chunking, chunks, retriever in (("document-aware", doc_chunks, doc_retriever),
                                    ("fixed-size", fixed_chunks, fixed_retriever)):
    for name, index in retriever.named().items():
        grades[(chunking, name)] = rag.evaluate_retriever(index, chunks, gold)

table = pd.DataFrame([{"chunking": c, "retriever": n,
                       **{f"r{k}": v["recall_at_k"][str(k)] for k in GATE["retrieval_ks"]},
                       "MRR": v["mean_reciprocal_rank"]}
                      for (c, n), v in grades.items()])
print(table.to_string(index=False))

recorded = {("document-aware", n): REC["rag"]["retrieval"]["document_aware"][n] for n in
            ("tf-idf", "bm25", "fusion")}
recorded |= {("fixed-size", n): REC["rag"]["retrieval"]["fixed_size"][n] for n in
             ("tf-idf", "bm25", "fusion")}
ok = (len(grades) == GATE["retrieval_combinations"]
      and all(v["n_questions"] == GATE["retrieval_graded_questions"] for v in grades.values())
      and all(v["depth"] == GATE["retrieval_depth"] for v in grades.values())
      and all(round(grades[key]["mean_reciprocal_rank"], GATE["retrieval_decimals"])
              == round(recorded[key]["mean_reciprocal_rank"], GATE["retrieval_decimals"])
              and grades[key]["recall_at_k"] == recorded[key]["recall_at_k"]
              and grades[key]["ranks"] == recorded[key]["ranks"]
              for key in grades))
gate("B4", ok, f"all {len(grades)} chunking x retriever combinations reproduce the recorded run — every "
               f"recallk, every MRR and all {GATE['retrieval_graded_questions']} per-question ranks")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6), sharey=True)
ks = GATE["retrieval_ks"]
for ax, chunking in zip(axes, ("document-aware", "fixed-size")):
    for name, colour, marker in (("tf-idf", MUTED, "s"), ("bm25", TEAL, "o"), ("fusion", RED, "^")):
        y = [grades[(chunking, name)]["recall_at_k"][str(k)] for k in ks]
        ax.plot(ks, y, marker=marker, color=colour, label=name, linewidth=1.6, markersize=5)
    ax.set_title(f"{chunking} chunking", fontsize=10)
    ax.set_xlabel("k"); ax.set_xticks(ks); ax.set_ylim(0.7, 1.02); ax.grid(alpha=0.25)
axes[0].set_ylabel("recallk"); axes[0].legend(frameon=False, fontsize=8, loc="lower right")
fig.suptitle("Where the two chunkings differ is at k = 1, and only at k = 1", fontsize=11)
fig.tight_layout(); plt.show()

d, f = grades[("document-aware", "bm25")], grades[("fixed-size", "bm25")]
print(f"BM25, rank 1:    document-aware {d['recall_at_k']['1']:.2f}  against fixed-size {f['recall_at_k']['1']:.2f}")
print(f"BM25, recall5:  document-aware {d['recall_at_k']['5']:.2f}  against fixed-size {f['recall_at_k']['5']:.2f}")

### Reading the table: two findings, and the second one is the interesting one

**Document-aware chunking wins on rank 1, and fixed-size catches up by recall5.** BM25 puts the right
statement first for 95 per cent of the questions on document-aware chunks and 80 per cent on fixed-size ones;
by $k = 5$ the fixed-size index has caught up and passed it (1.00 against 0.95). The difference is not that
fixed-size chunking finds the wrong documents. It is that a 500-character window starting in an arbitrary
place dilutes each chunk's match, so the right statement arrives at rank 3 instead of rank 1. If a reader
sees five passages, that costs little. If a reader sees one, it costs everything.

**Fusion does not beat BM25 here.** Mean reciprocal rank 0.933 against 0.958; on rank 1, 0.90 against 0.95.
This is the opposite of what the method is usually sold on, and the mechanism is the one the four-document
table above showed: **RRF reads ranks only**. TF-IDF and BM25 are not two independent views of this corpus — they
share a vocabulary, a tokeniser and a term space, and they mostly agree. Where they differ, BM25 is the
better of the two, and averaging ranks with a weaker input drags the stronger one down. Fusion buys
robustness against a retriever that is catastrophically wrong on some queries. It has nothing to sell when
one input is simply better than the other everywhere. Deck 6.2 merges a dense list with a keyword list, two
views that fail in different places, which is the case fusion is built for; here both lists count words.
Exercise 2(e) runs the deck's version on B1's passages.

**What to do with that.** Report it. A lab that ran fusion, found it lost, and quietly reported BM25 as
"our retriever" would have published a true number from an undisclosed search over methods. The gate on this
block compares all six MRRs against the recorded run — fusion's included — so it would pass unchanged if
fusion had won.

The rest of the notebook uses the fused retriever anyway, for one reason that has nothing to do with
ranking: `evidence_score` needs a **cosine**, and the fused object carries the TF-IDF index that produces one.

# Block C — Retrieve, answer, or refuse

A retriever always returns something. Ranked lists have no empty state: ask a corpus of FOMC statements what
the Bank of Japan decided and it will hand back its five closest paragraphs, in confident order, with no
indication that none of them is an answer. Block D makes refusal a **number** rather than an instruction.

## C1. The six questions this corpus cannot answer

They are deliberately near misses. Four FOMC statements name the Bank of Japan in swap-line paragraphs, so
the *words* are in the corpus; what the Bank of Japan decided is not, and never could be.

In [ ]:
print(f"{'id':<5}{'kind':<16}question")
for q in gold:
    if not q.answerable:
        print(f"{q.id:<5}{q.kind:<16}{q.question}")
        print(f"{'':<21}{textwrap.fill(q.note, 92, subsequent_indent=' ' * 21)}")
print(f"\n{GATE['gold_in_corpus']} answerable questions carry {len(set(a for q in gold for a in q.anchors))} "
      f"distinct anchor statements between them, 1994 to 2026.")

## C2. A refusal that is a number, not a sentence in a prompt

The statistic is the **maximum TF-IDF cosine among the top-$k$ fused chunks** — the strongest evidence the
answer would be allowed to quote. A cosine and not the fused rank score, because the fused score depends only
on ranks and so is nearly the same number for every query: it cannot tell a query the corpus answers from one
it does not. A cosine can. It is bounded, comparable across queries, and it falls when a query's distinctive
terms are missing from the corpus altogether.

Refusing is the positive class. **Precision** is the share of refusals that were right; **recall** is the
share of unanswerable questions that were turned away. The threshold maximises F1, ties broken in favour of
answering more answerable questions — because a system that refuses everything scores perfect recall and is
useless. Candidates are the midpoints between neighbouring observed scores, which is the whole of the search
space: nowhere else can a threshold change a decision.

In [ ]:
calibration = rag.calibrate_refusal(doc_retriever, gold)
threshold = calibration["threshold"]
scores = calibration["scores"]
chosen, kinds = calibration["chosen"], calibration["score_by_kind"]

print(f"statistic: {calibration['statistic']}, top_k = {calibration['top_k']}\n")
print(f"{'':<16}{'n':>4}{'minimum':>10}{'median':>9}{'maximum':>10}")
for kind in ("in_corpus", "out_of_corpus"):
    d = kinds[kind]
    print(f"{kind:<16}{d['n']:>4}{d['minimum']:>10.3f}{d['median']:>9.3f}{d['maximum']:>10.3f}")
print(f"\nthreshold {threshold:.6f}   refusals {chosen['refusals']}   precision {chosen['precision']:.3f}"
      f"   recall {chosen['recall']:.3f}   F1 {chosen['f1']:.3f}")
print(f"answerable questions kept {chosen['answerable_kept']} of {GATE['gold_in_corpus']}, "
      f"refused {chosen['answerable_refused']}")

ok = (round(threshold, GATE["refusal_decimals"]) == GATE["refusal_threshold"]
      and chosen["refusals"] == GATE["refusal_refusals"]
      and chosen["precision"] == GATE["refusal_precision"]
      and chosen["recall"] == GATE["refusal_recall"]
      and chosen["f1"] == GATE["refusal_f1"]
      and chosen["answerable_kept"] == GATE["refusal_answerable_kept"]
      and calibration["statistic"] == GATE["refusal_statistic"]
      and calibration["scores"] == REC["rag"]["refusal"]["calibration"]["scores"])
gate("C1", ok, f"threshold {threshold:.6f} and all {len(calibration['scores'])} evidence scores reproduce "
               f"the recorded run; precision {chosen['precision']:.3f}, recall {chosen['recall']:.3f}, "
               f"F1 {chosen['f1']:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 3.0))
for kind, colour, y in (("in_corpus", TEAL, 1.0), ("out_of_corpus", RED, 0.0)):
    xs = [scores[q.id] for q in gold if (q.kind == kind)]
    ax.scatter(xs, [y] * len(xs), s=44, color=colour, zorder=3,
               label=f"{kind.replace('_', '-')} ({len(xs)})")
    for q in gold:
        if q.kind == kind and q.id in GATE["refusal_slip_through_ids"]:
            ax.annotate(q.id, (scores[q.id], y), textcoords="offset points", xytext=(0, 12),
                        ha="center", fontsize=8, color=RED)
ax.axvline(threshold, color=MUTED, linestyle="--", linewidth=1.4)
ax.annotate(f"threshold {threshold:.3f}", (threshold, 0.5), textcoords="offset points",
            xytext=(6, 0), fontsize=8, color=MUTED)
ax.set_yticks([0, 1]); ax.set_yticklabels(["cannot be\nanswered", "can be\nanswered"], fontsize=8)
ax.set_xlabel("evidence score — maximum TF-IDF cosine among the top 5 fused chunks")
ax.set_ylim(-0.5, 1.5); ax.legend(frameon=False, fontsize=8, loc="center right")
ax.set_title("The two classes overlap, and the overlap is the whole subject", fontsize=11)
fig.tight_layout(); plt.show()

## C3. What the threshold costs

An answerable question that is refused counts as a **miss at every k**. That is the honest way to price a
refusal: the corpus had the answer, the pipeline declined to give it. So price it.

In [ ]:
cost = rag.refusal_cost(doc_retriever, doc_chunks, gold, threshold)
print(f"{'':<10}" + "".join(f"{'r' + str(k):>9}" for k in GATE["retrieval_ks"]) + f"{'MRR':>9}")
print(f"{'before':<10}" + "".join(f"{cost['recall_before'][str(k)]:>9.3f}" for k in GATE["retrieval_ks"])
      + f"{cost['mrr_before']:>9.4f}")
print(f"{'after':<10}" + "".join(f"{cost['recall_after'][str(k)]:>9.3f}" for k in GATE["retrieval_ks"])
      + f"{cost['mrr_after']:>9.4f}")
print(f"\n{cost['answerable_refused']} of {cost['answerable']} answerable questions were refused"
      f"{': ' + ', '.join(cost['refused_ids']) if cost['refused_ids'] else '.'}")
print("On this gold set the threshold costs nothing in recall. That is a fact about this corpus and these")
print("26 questions, not a property of thresholds: the in-corpus minimum is 0.237 and the threshold is")
print("0.218, so the margin is 0.019 — one badly worded answerable question would close it.")

recorded_cost = REC["rag"]["refusal"]["cost"]
ok = (cost["recall_after"] == recorded_cost["recall_after"]
      and cost["recall_before"] == recorded_cost["recall_before"]
      and round(cost["mrr_after"], 4) == round(recorded_cost["mrr_after"], 4)
      and cost["answerable_refused"] == recorded_cost["answerable_refused"]
      and cost["refused_ids"] == recorded_cost["refused_ids"])
gate("C2", ok, f"recallk and MRR before and after the threshold both reproduce the recorded run "
               f"({cost['answerable_refused']} answerable questions refused, as recorded)")

## C4. The two that get through, and why a lexical threshold cannot tell 1987 from 1994

Precision 1.000 means every refusal was right. Recall 0.667 means **two of the six unanswerable questions
scored above the threshold and were answered**. Look at which two, and at what the retriever did with the
part of the question that should have stopped it.

In [ ]:
for q in gold:
    if q.answerable or scores[q.id] < threshold:
        continue
    unseen = doc_retriever.space.unseen_terms(q.question)
    answer = rag.grounded_answer(q.question, doc_retriever, threshold, first=statements[0].date,
                                 last=statements[-1].date)
    print(f"{q.id}  score {scores[q.id]:.3f}  against threshold {threshold:.3f}")
    print(textwrap.fill(f"   {q.question}", 100, subsequent_indent="   "))
    print(f"   terms outside the fitted vocabulary, contributing exactly zero: {unseen}")
    print(f"   answered with: {answer.evidence[0].date}, {answer.evidence[0].section}")
    print()

print("q24 asks about the meeting of August 12, 1987. The corpus begins in 1994, so '1987' is not in the")
print("vocabulary — and a term outside the vocabulary contributes NOTHING to a cosine. It cannot push the")
print("score down, because there is no column for it to be absent from. What is left of the question is")
print("'what did the Committee decide about the target for the federal funds rate', which is as ordinary")
print("as FOMC vocabulary gets, so the score is ordinary too. A lexical threshold measures vocabulary")
print("overlap. The year is the only thing that makes this question unanswerable, and the year is exactly")
print("what the threshold cannot see.")
print()
print("q22 fails the other way: 'European Central Bank' and 'asset purchase programme' are ALL in this")
print("corpus, in statements about the Federal Reserve's own purchases and its swap lines. The words are")
print("present; the institution's decisions are not. No threshold on word overlap separates those.")
print()
print("The fix is not a better threshold. It is a date filter the retriever applies before it scores —")
print("metadata as a hard constraint, not as five more tokens in a bag of words. B1's one-line filter on the")
print("address is that fix in its simplest form; exercise 3 builds it properly.")

ok = (sorted(q.id for q in gold if not q.answerable and scores[q.id] >= threshold)
      == GATE["refusal_slip_through_ids"]
      and GATE["refusal_slip_through_year"] in doc_retriever.space.unseen_terms(
          next(q.question for q in gold if q.id == "q24"))
      and GATE["refusal_slip_through_year"] not in doc_retriever.space.vocabulary)
gate("C3", ok, f"the questions above the threshold are {GATE['refusal_slip_through_ids']}, as recorded; "
               f"'{GATE['refusal_slip_through_year']}' is outside the fitted vocabulary and so contributes "
               f"zero to every score")

## C5. Three failures, reproduced on purpose

Every one of these is a real failure mode with a real fix, and every one is invisible from the output alone.

### C5a. A stale index — the failure with no symptom

In [ ]:
stale = rag.failure_stale_index(statements, threshold=threshold)
print(f"question: {stale['question']}\n")
print(f"{'':<8}{'indexed':>9}{'newest':>13}{'anchor in index':>17}{'anchor rank':>13}{'cosine':>9}")
for label in ("before", "after"):
    s = stale[label]
    print(f"{label:<8}{s['statements_indexed']:>9}{s['newest_document']:>13}"
          f"{str(s['anchor_in_index']):>17}{str(s['anchor_rank']):>13}{s['top_cosine']:>9.3f}")
print(f"\nthe stale index returns: {stale['before']['top_dates'][0]} "
      f"(the question asks about {stale['anchor']})")
print(textwrap.fill(f"   {stale['before']['answer'][:220]}...", 100, subsequent_indent="   "))
print("\nThe rebuilt index puts the right statement first. Nothing in the stale index's output says it is")
print("stale: the refusal fired here only because the cosine happened to fall below the threshold, and it")
print("would not have on a question whose vocabulary the old corpus shares. A refresh schedule is part of")
print("a retrieval system's accuracy, not an operational detail.")

### C5b. A chunk boundary through the middle of the word "decided"

In [ ]:
boundary = rag.failure_chunk_boundary(statements, threshold=threshold)
print(f"the decision sentence of {boundary['date']}, {boundary['sentence_characters']} characters:")
print(textwrap.fill(f"   {boundary['sentence']}", 100, subsequent_indent="   "))
print(f"\na fixed-size boundary falls after character {boundary['cut_after_characters']}, "
      f"inside a word: {boundary['cut_inside_a_word']}")
print(f"   chunk 1 ends   ...{boundary['first_half'][-46:]}|")
print(f"   chunk 2 begins |{boundary['second_half'][:46]}...")
print(f"\n{boundary['decision_sentences_cut']} decision sentences in this corpus are cut this way.")
print(textwrap.fill(boundary["reading"], 100))

ok = (boundary["date"] == GATE["boundary_date"]
      and boundary["sentence_characters"] == GATE["boundary_sentence_characters"]
      and boundary["cut_after_characters"] == GATE["boundary_cut_after_characters"]
      and boundary["cut_inside_a_word"] is GATE["boundary_cut_inside_a_word"]
      and boundary["decision_sentences_cut"] == GATE["boundary_decision_sentences_cut"]
      and boundary["sentences_cut"] == GATE["sentences_cut_by_fixed"]
      and stale["before"]["statements_indexed"] == GATE["stale_statements_before"]
      and stale["before"]["anchor_rank"] == GATE["stale_anchor_rank_before"]
      and stale["after"]["anchor_rank"] == GATE["stale_anchor_rank_after"])
gate("C4", ok, f"the stale index cannot reach {GATE['stale_anchor']} at any rank and the rebuilt one puts "
               f"it first; the boundary cuts {GATE['boundary_date']}'s decision sentence after "
               f"{GATE['boundary_cut_after_characters']} of {GATE['boundary_sentence_characters']} characters")

### C5c. An ambiguous query, answered confidently from the wrong meeting

The Committee raised the target range to 1/4 to 1/2 percent **twice**: lifting off the zero bound on
2015-12-16, and again on 2022-03-16 after the pandemic cuts. A question that describes the action without
naming the meeting fits both perfectly.

In [ ]:
amb = rag.failure_ambiguous_query(statements, threshold=threshold)
print(f"the phrase '{amb['phrase']}' appears in {amb['n_statements_containing_the_phrase']} statements:")
print("   " + "  ".join(amb["statements_containing_the_phrase"]))
for label in ("before", "after"):
    a = amb[label]
    print(f"\n{label}: {textwrap.fill(a['query'], 96, subsequent_indent='        ')}")
    print(f"   top dates {a['top_dates'][:3]}   intended meeting rank {a['intended_meeting_rank']}"
          f"   margin {a['margin']:.3f}")
print(f"\nUndated, the meeting the user meant ranks {amb['before']['intended_meeting_rank']}th, and the")
print("pipeline reports nothing unusual while answering from the wrong year. Adding the date puts it first.")
print("A retriever's confidence is a statement about vocabulary overlap, not about whether the right")
print("document was found. The fix is a question that says which meeting — which is why every question in")
print("the gold set carries a date.")

ok = (amb["before"]["intended_meeting_rank"] == GATE["ambiguous_rank_before"]
      and amb["after"]["intended_meeting_rank"] == GATE["ambiguous_rank_after"]
      and amb["n_statements_containing_the_phrase"] == GATE["ambiguous_carriers"]
      and amb["intended"] == GATE["ambiguous_intended"]
      and not amb["before"]["intended_meeting_first"] and amb["after"]["intended_meeting_first"])
gate("C5", ok, f"undated, the first chunk from {GATE['ambiguous_intended']} arrives at rank "
               f"{amb['before']['intended_meeting_rank']}; dated, at rank "
               f"{amb['after']['intended_meeting_rank']} — with the phrase carried by "
               f"{GATE['ambiguous_carriers']} meetings either way")

## C6. Hand the reader the right passage

Everything above is retrieval. Retrieval finds the paragraph; something still has to **read** it. Session 1's
Machine B reads 128 characters, which is not enough to hold an answer sentence and a passage at once, so a
second character model was trained here with the context doubled to 256 and nothing else changed. It is
loaded in eval mode and only ever measured.

Two experiments, both scoring the same answer sentence, both putting it at **exactly the same positions** in
the window, so only the context differs:

- **passage** — the model is given the passage retrieval actually returned, with the answer sentence itself
  cut out of it, and must predict the answer sentence. The control is a passage from a meeting at least ten
  years away.
- **verbatim** — the answer sentence is placed in the window **twice**, and only the second copy is scored.
  The control puts unrelated text in the first half. This is the easiest possible use of a context: copy what
  you were just shown. A model that reads its window scores near zero here.

Losses are in **nats per character**. For scale, a model that has learned nothing scores $\ln 78 = 4.357$,
and this reader's held-out validation loss is 1.305.

In [ ]:
reader = rag.load_reader()                   # four file hashes checked against reader/MANIFEST.json
print(f"reader: {reader.parameters:,} parameters, context {reader.model.context}, "
      f"vocabulary {len(reader.vocabulary)}")
print(f"        trained {reader.manifest['training']['steps']} steps, held-out validation loss "
      f"{reader.loss_table['final_validation']:.4f}, uniform baseline "
      f"{reader.loss_table['uniform_loss']:.4f}\n")

experiment = rag.reader_experiment(reader, doc_retriever, statements, gold)
p, v = experiment["passage"], experiment["verbatim"]
print(f"{'experiment':<12}{'treated':>10}{'control':>10}{'gap':>8}{'share':>9}{'helped':>9}")
print(f"{'passage':<12}{p['retrieved']:>10.4f}{p['unrelated']:>10.4f}{p['gap']:>8.4f}"
      f"{100 * p['gap_share_of_unrelated']:>8.1f}%{p['helped']:>6} / {experiment['n_questions']}")
print(f"{'verbatim':<12}{v['copy']:>10.4f}{v['control']:>10.4f}{v['gap']:>8.4f}"
      f"{100 * v['gap_share_of_control']:>8.1f}%{v['helped']:>6} / {experiment['n_questions']}")

recorded_reader = REC["rag"]["reader"]["experiment"]
d = GATE["reader_loss_decimals"]
ok = (reader.parameters == GATE["reader_parameters"]
      and reader.model.context == GATE["reader_context"]
      and sorted(r["file"] for r in reader.manifest["files"]) == sorted(GATE["reader_files"])
      and experiment["n_questions"] == GATE["reader_questions"]
      and experiment["target_characters"] == GATE["reader_target_characters"]
      and all(round(experiment[k][f], d) == round(recorded_reader[k][f], d)
              for k, f in (("passage", "retrieved"), ("passage", "unrelated"),
                           ("verbatim", "copy"), ("verbatim", "control")))
      and all(row["target_characters"] <= GATE["reader_target_characters"]
              for row in experiment["rows"]))
gate("C6", ok, f"the reader's {len(GATE['reader_files'])} files hash-check against its manifest at "
               f"{reader.parameters:,} parameters; all four recorded losses reproduce to "
               f"{d} decimals, whatever they say")

In [ ]:
rows = pd.DataFrame(experiment["rows"])
fig, ax = plt.subplots(figsize=(8.6, 3.2))
order = rows["verbatim_control"].sort_values().index
x = np.arange(len(rows))
ax.plot(x, rows.loc[order, "verbatim_control"], marker="o", markersize=4, color=MUTED, linewidth=1.2,
        label="verbatim control (unrelated text in the first half)")
ax.plot(x, rows.loc[order, "verbatim_copy"], marker="o", markersize=4, color=TEAL, linewidth=1.2,
        label="verbatim copy (the same sentence, twice)")
top = float(rows[["verbatim_control", "verbatim_copy"]].to_numpy().max())
ax.axhline(0, color=RED, linestyle="--", linewidth=1.0)
ax.annotate("what a model that reads its window would score", (0.2, 0.05 * top), fontsize=8, color=RED)
ax.set_xlabel("the 20 answerable questions, ordered by control loss")
ax.set_ylabel("nats per character"); ax.set_ylim(-0.08 * top, 1.12 * top)
ax.set_xticks(range(0, len(rows), 2))
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title("Copy the sentence you were just shown: 1.523 against 1.570", fontsize=11)
fig.tight_layout(); plt.show()

### Retrieval without a reader is not grounding

The answer sentence is in the window, character for character, and the model still spends 1.523 nats per
character predicting it against 1.570 with unrelated text there instead. **Copying text it was just shown
buys three per cent.** The retrieved passage buys the same three per cent.

Read that carefully, because it is easy to draw the wrong conclusion from it.

- It is **not** a failure of retrieval. The right passage was found, at rank 1, and handed over. Every number
  in Block B stands.
- It is what "the reader is the other half of the system" costs when the reader is a three-layer character
  model with 648 thousand parameters. Copying from context is an emergent behaviour of scale: it appears in
  models orders of magnitude larger than this one, and it is absent here.
- So the pipeline you have built is **half a system**, and the half you built is measurable on your laptop
  while the other half is not. That asymmetry is worth naming out loud, because it is the same asymmetry
  that lets a paper report retrieval metrics and call the result grounded.

No gate in this notebook requires this gap to be large. Gate D6 checks the four losses against the recorded
run and the procedure that produced them; it would pass unchanged at a gap of zero or of ten times this size.

# Block D — Faithfulness, by hand

recallk says the right document was shown. It says nothing about whether the **answer** used it. Between
the two sits faithfulness: is every claim in this answer supported by the text that was retrieved?

The method is manual and it does not scale, which is the point of doing it once by hand. Break the answer
into **atomic claims** — a claim is atomic when it can be marked supported or not without splitting it
further — and mark each against the retrieved text. Three verdicts, not two:

| verdict | meaning |
|---|---|
| **supported** | the retrieved text says this |
| **unsupported** | the retrieved text does not say this — which is not the same as saying it is false |
| **contradicted** | the retrieved text says something incompatible with it |

The distinction between *unsupported* and *contradicted* is the one that does the work. A true claim the
retrieved text does not support is still a faithfulness failure, because the citation attached to it is a
citation to something else.

In [ ]:
task_question = "what did the Committee say about inflation at the meeting of March 16, 2022"
machine = rag.grounded_answer(task_question, doc_retriever, threshold,
                              first=statements[0].date, last=statements[-1].date)
retrieved_text = "\n".join(doc_chunks[c["chunk_id"]].text for c in machine.considered)

print(f"the pipeline's own answer, refused = {machine.refused}, evidence score {machine.score:.3f}\n")
print(textwrap.fill(machine.text, 100, subsequent_indent="   "))
print(f"\nit quoted {len(machine.evidence)} sentences out of the {len(machine.considered)} chunks it")
print("considered. The answer is EXTRACTIVE: nothing was paraphrased, so every word of it can be found by")
print("searching the retrieved text. Check that, rather than believing it.")
for e in machine.evidence:
    print(f"   found verbatim in chunk {e.chunk_id} ({e.date}): {e.sentence in retrieved_text}")
print("\nAnd read what it chose. Two of those three sentences do not answer a question about inflation:")
print("one is about the balance sheet and one is about who voted as an alternate member. Sentences are")
print("ranked on vocabulary overlap with the question plus a share of their chunk's score, and every")
print("sentence in this statement carries the same date and section fingerprint. EXTRACTIVE AND CITED IS")
print("NOT THE SAME AS RESPONSIVE — which is the second thing recallk cannot see.")

## D1. An answer that was not extractive

Now the realistic case. Below is an answer of the kind a language model writes for this task: fluent, mostly
right, carrying a citation. Six atomic claims. Mark each one against `retrieved_text` and nothing else — not
against what you know about March 2022.

In [ ]:
UNDER_REVIEW = (
    "At its meeting of March 16, 2022 the Committee said that inflation remains elevated, reflecting "
    "supply and demand imbalances related to the pandemic, higher energy prices, and broader price "
    "pressures. It decided to raise the target range for the federal funds rate to 1/4 to 1/2 percent. "
    "It described the invasion of Ukraine by Russia as causing tremendous human and economic hardship. "
    "The Committee expects inflation to return to its 2 percent objective by the end of 2023. The "
    "ten-year Treasury yield rose 57 basis points over the following month. The vote was unanimous."
)

# Atomic claims, and the span of the claim that has to be found for it to count as supported.
CLAIMS = [
    ("inflation remains elevated, reflecting supply and demand imbalances",
     "inflation remains elevated, reflecting supply and demand imbalances"),
    ("the target range was raised to 1/4 to 1/2 percent",
     "raise the target range for the federal funds rate to 1/4 to 1/2 percent"),
    ("the invasion of Ukraine is causing tremendous human and economic hardship",
     "invasion of Ukraine by Russia is causing tremendous human and economic hardship"),
    ("inflation is expected to return to 2 percent BY THE END OF 2023",
     "by the end of 2023"),
    ("the ten-year Treasury yield rose 57 basis points over the following month",
     "57 basis points"),
    ("the vote was unanimous",
     "unanimous"),
]

print(textwrap.fill(UNDER_REVIEW, 100))
print(f"\nchecked against {len(machine.considered)} retrieved chunks, "
      f"{len(retrieved_text):,} characters\n")
print(f"{'#':<3}{'in retrieved text':<19}claim")
found = []
for i, (claim, span) in enumerate(CLAIMS, start=1):
    hit = span.lower() in retrieved_text.lower()
    found.append(hit)
    print(f"{i:<3}{('yes' if hit else 'NO'):<19}{claim}")

## D2. The marking, and why three of the six fail differently

String search is the mechanical half. The verdict is yours, and the three that fail here fail for three
different reasons — which is exactly why "hallucination rate" as a single number tells you nothing.

In [ ]:
VERDICTS = [
    ("supported",    "verbatim in the statement of 2022-03-16, first paragraph."),
    ("supported",    "verbatim in the decision paragraph."),
    ("supported",    "verbatim in the second paragraph."),
    ("unsupported",  "the statement says the Committee expects inflation to return to its 2 percent "
                     "objective. It names NO DATE. 'by the end of 2023' came from somewhere else — the "
                     "projections released the same day, a different document. Splitting the claim is what "
                     "exposes it: the first half is supported, the second half is not, and a compound claim "
                     "that is half supported is unsupported."),
    ("unsupported",  "true, and not in any FOMC statement. It comes from the yield panel, which retrieval "
                     "never touched. An answer citing the statement for this number cites the wrong source; "
                     "Block A got it from a tool and Block E verifies it."),
    ("contradicted", "the statement records that James Bullard voted against, preferring 0.5 percentage "
                     "point. The retrieved text says the opposite of the claim."),
]

print(f"{'#':<3}{'verdict':<15}{'span found':<12}why")
for i, ((claim, span), (verdict, why)) in enumerate(zip(CLAIMS, VERDICTS), start=1):
    print(f"{i:<3}{verdict:<15}{str(found[i - 1]):<12}{textwrap.fill(why, 68, subsequent_indent=' ' * 30)}")

counts = collections.Counter(v for v, _ in VERDICTS)
print(f"\n{counts['supported']} of {len(CLAIMS)} claims supported; {counts['unsupported']} unsupported, "
      f"{counts['contradicted']} contradicted.")
print("The mechanical column says NO for claims 4, 5 and 6 alike, and they are three different failures.")
print("A string search can find support. It cannot find contradiction: the word 'unanimous' is absent from")
print("the retrieved text, and so is 'by the end of 2023', and only reading tells you that one of those")
print("absences sits beside a sentence saying the opposite.")
print()
print("Faithfulness is not accuracy. Claim 5 is TRUE and unfaithful; claim 6 is false and contradicted;")
print("claim 4 is a true sentence with a false ending welded on. One number over the six would hide all")
print("three distinctions, and the three have different fixes: a tool, a citation policy, and a reader")
print("that can be trusted to read a vote paragraph.")

ok = (len(CLAIMS) == GATE["faithfulness_claims"] == len(VERDICTS)
      and len(machine.evidence) <= GATE["faithfulness_max_sentences"]
      and all(e.sentence in retrieved_text for e in machine.evidence)
      and all(e.date and e.chunk_id is not None for e in machine.evidence)
      and found == [True, True, True, False, False, False])
gate("D1", ok, f"every sentence the pipeline quoted is found verbatim in a retrieved chunk and carries its "
               f"chunk id and date; all {len(CLAIMS)} claims were marked against the retrieved text alone, "
               f"the span search agreeing with the hand verdicts")

**Your turn, before Block E.** Run `rag.grounded_answer` on a question of your own about a meeting you know
— a 2008 meeting is a good test, since the language is unusual — and mark its output the same way. Two things
to watch for: an extractive answer can still be unfaithful when a sentence is quoted **out of the section it
belongs to** (a "guidance" sentence read as a decision), and a question with no date will hand you the right
sentences from the wrong meeting, as Block C5c showed.

# Block E — Govern it

Three rules, all of them code, none of them a sentence in a prompt. A budget is a number the loop checks
before it sends. A permission is a list the registry checks before it calls. A verifier is a function that
recomputes what the answer claimed. Block A exercised all three inside the loop; Block E takes them apart.

## E1. The verifier: recompute every number the answer asserted

In [ ]:
verification = run["verification"]
print(f"the answer, verified against the panel for {verification['meeting_date']}:\n")
print(f"{'claim':<24}{'stated':>12}{'expected':>12}   verdict")
for c in verification["checks"]:
    print(f"{c['claim']:<24}{str(c['stated']):>12}{str(c['expected']):>12}   {c['verdict']}")
print(f"\n{verification['n_checks']} checks: {verification['counts']}; passes = {verification['passes']}")

bad = ag.unverifiable_answer_demo()
print(f"\nthe same answer with one number changed, and with the window sentence removed:\n")
print(f"{'variant':<18}{'checks':>8}   counts")
for name, result in bad.items():
    print(f"{name:<18}{result['n_checks']:>8}   {result['counts']}   passes = {result['passes']}")
print("\n'absent' is not 'passes'. An answer that quietly drops the window its numbers were measured over")
print("has not been verified; it has been skirted. A verifier that treated a missing claim as a passing")
print("one would reward exactly the answer that is hardest to check.")

## E2. A gate that is a valid regular expression and stops nothing

This is a real bug, found on 12 September 2026. A `PreToolUse` hook was written to require a source column on
any CSV the agent wrote:

```json
{"hooks": {"PreToolUse": [{"matcher": "Write(data/*.csv)",
                           "hooks": [{"type": "command", "command": "python3 checks/require_source_column.py"}]}]}}
```

It never fired. `matcher` is compared with the **tool name** and nothing else: a plain name exactly, and
anything with other characters in it (parentheses, a slash, a star) as a regular expression that may match
anywhere in the name. `Write(data/*.csv)` is a perfectly valid regular expression; it just matches no tool
name that exists, so the group never activates and nothing is ever run. Nothing errors, nothing warns, and the
settings file reads exactly like a rule that is in force.

The argument pattern belongs in the individual hook's `if` field, which takes permission-rule syntax: there,
`Write(data/*.csv)` means a Write to a CSV file under `data/` (Claude Code's hooks reference, read on
20 October 2026). Four probes tell the two forms apart.

In [ ]:
report = ag.gate_report()
print(f"{'':<10}{'matcher':<22}{'if':<22}{'valid regex':>12}{'fired':>7}{'blocked':>9}")
for label in ("broken", "working"):
    s = report["settings"][label]
    print(f"{label:<10}{s['matcher']:<22}{(s['if'] or '—'):<22}"
          f"{str(s['matcher_is_valid_regex']):>12}{s['fired']:>7}{s['blocked']:>9}")
print()
print(f"{'probe':<44}{'broken':>10}{'working':>20}")
for b, w in zip(report["settings"]["broken"]["probes"], report["settings"]["working"]["probes"]):
    verdict = "blocked" if w["blocked"] else ("fired, allowed" if w["hook_fires"] else "not matched")
    print(f"{b['label']:<44}{('fired' if b['hook_fires'] else 'not matched'):>10}{verdict:>20}")
print(f"\n{textwrap.fill(report['verdict'], 100)}")
print(f"\nwhen the working gate blocks, it says why:")
blocking = next(p for p in report["settings"]["working"]["probes"] if p["blocked"])
print(f"   {blocking['reason']}")

ok = (len(report["settings"]["broken"]["probes"]) == GATE["hook_probes"]
      and report["settings"]["broken"]["fired"] == GATE["hook_broken_fired"]
      and report["settings"]["broken"]["blocked"] == GATE["hook_broken_blocked"]
      and report["settings"]["working"]["fired"] == GATE["hook_working_fired"]
      and report["settings"]["working"]["blocked"] == GATE["hook_working_blocked"]
      and report["settings"]["broken"]["matcher_is_valid_regex"] is GATE["hook_broken_is_valid_regex"]
      and report["settings"]["working"]["matcher_is_valid_regex"] is GATE["hook_working_is_valid_regex"]
      and not report["settings"]["broken"]["matcher_matches_any_tool_name"])
gate("E1", ok, f"over {GATE['hook_probes']} probes the broken matcher fires {GATE['hook_broken_fired']} "
               f"times while compiling as a valid regular expression; the working form fires "
               f"{GATE['hook_working_fired']} and blocks {GATE['hook_working_blocked']}")

## E3. Reliability: the same agent, the same task, run again

Three scenarios recorded for this course's Summer 2026 agent lab, which is still on the course site, ran the
same agents over the same tasks repeatedly; their eight recorded summaries ship in `session06_assets/lec09_summaries/`.
Two numbers come out of them, and the second is the one nobody reports.

**Agreement** — whether repeated runs find the same defects — can only be bounded from the recorded
summaries, which hold per-defect counts out of $k$ runs rather than each run's finding set. The bounds
coincide whenever every defect was found by all $k$ runs or by none.

**Completion** — whether the harness finished at all — needs no bounding. It is counted.

In [ ]:
rel = ag.reliability()
print(f"scenarios: {', '.join(rel['scenarios'])}\n")
print(f"runs attempted        {rel['runs_attempted']}")
print(f"runs completed        {rel['runs_completed']}")
print(f"runs the harness could not finish  {rel['runs_failed']}   "
      f"({100 * rel['harness_failure_rate']:.0f}% of attempts)")
print(f"\ncells with repeats    {rel['n_cells_with_repeats']} of {rel['n_cells']}")
print(f"unanimous             {rel['unanimous_cells']} of {rel['n_cells_with_repeats']}")
print(f"same-answer rate      {rel['mean_same_answer_lower_rate']:.4f} to "
      f"{rel['mean_same_answer_upper_rate']:.4f} (lower and upper bounds)")
print(f"\n{rel['runs_completed']} of {rel['runs_attempted']} attempted runs completed. Before a single")
print("question about whether the answers agree, a quarter of the attempts produced no answer to compare.")
print("A reported agreement rate computed over the runs that finished is conditioned on finishing, and")
print("that conditioning is almost never stated.")

redaction = ag.check_redaction()
print("\ntwo real agent runs of 12-13 September 2026 ship beside the module as redacted transcripts;")
print("the redaction is checked, not asserted:")
print(f"   {redaction['n_files']} files, clean = {redaction['clean']}, "
      f"{len(redaction['rules'])} ordered rules, {sum(redaction['totals'].values())} surviving matches")

recorded_rel = REC["agent"]["reliability"]
ok = (rel["runs_attempted"] == GATE["reliability_runs_attempted"] == recorded_rel["runs_attempted"]
      and rel["runs_completed"] == GATE["reliability_runs_completed"]
      and rel["runs_failed"] == GATE["reliability_runs_failed"]
      and rel["n_cells"] == GATE["reliability_cells"]
      and rel["n_cells_with_repeats"] == GATE["reliability_cells_with_repeats"]
      and rel["unanimous_cells"] == GATE["reliability_unanimous_cells"]
      and len(rel["scenarios"]) == GATE["reliability_scenarios"]
      and redaction["n_files"] == GATE["redaction_files"]
      and redaction["clean"] is GATE["redaction_clean"])
gate("E2", ok, f"{rel['runs_completed']} of {rel['runs_attempted']} attempted runs completed across "
               f"{len(rel['scenarios'])} recorded scenarios, reproducing the recorded reliability table; "
               f"{redaction['n_files']} transcripts re-checked for identifiers")

# Your turn

**1. Price the loop.** Using `ag.resend_cost` and `ag.price`, take the seven-turn ledger from Block A and ask
what the same task would cost at 20 turns and at 50, at rates you supply yourself and date in a comment.
Then find the cheapest intervention: fewer turns, smaller tool results, or summarising the message list every
$n$ turns, which is what Deck 6.1 calls compaction. Give the crossover $n$.

**2. Make fusion win, or show it cannot.** Fusion loses to BM25 here at MRR 0.933 against 0.958.
(a) Sweep $k$ in `rag.reciprocal_rank_fusion` over $\{1, 5, 20, 60, 200\}$ and plot MRR against it; at small
$k$ the top of the curve is steep and a first place counts for much more. (b) Try a weighted fusion, summing
$w_i / (k + \mathrm{rank})$ with the weight on BM25 above the weight on TF-IDF, and find the weight at which
fusion first matches BM25. (c) Now say what you have actually learned from (b) — you fitted a weight on the
same 20 questions you are grading on, and the honest report of that number needs a sentence you must write.
(d) Under what condition on the two retrievers would rank fusion beat both? State it in terms of where each
one's errors fall. (e) Deck 6.2 fuses a dense list with a keyword list, two views that fail in different
places. Do it on B1's 1,264 passages: B1's Step 5 already ranked them for each graded question by the toy's
matrix (`V @ Qg[k]`, the questions encoded by the same model) and counted a hit when a passage's address names
one of the question's anchor statements. Rank them again by `rag.BM25Index(docs)`, fuse the two rankings with
`rag.reciprocal_rank_fusion`, and count the same way for the 20 answerable questions. Does fusion beat both
here? Where do the dense list's misses fall, and how many of them are the wrong year?

**3. A date filter, and what it buys.** Block C4 showed that a lexical threshold cannot see the year: `1987`
is outside the fitted vocabulary, so it contributes zero and q24 scores like any ordinary question about the
federal funds rate. Add a hard constraint instead of a soft one. (a) Parse a four-digit year out of each
question and, when there is one, restrict the candidate chunks to statements from that year before scoring.
(b) Re-run `rag.calibrate_refusal` and report the new precision, recall and F1 on the same 26 questions.
(c) The filter should catch q24 and leave q22 exactly where it was — say why, in one sentence, before you run
it. (d) What does the filter do to a question whose date is written "last December"?

**4. The reader, made to matter.** Block C6's verbatim experiment is the floor: copying a sentence from the
window buys 3 per cent. (a) Re-run `rag.reader_experiment` with `target_chars` at 40 and at 200 and plot the
gap against target length — a shorter target is a larger share of the window. (b) Put the answer sentence at
the very start of the prefix rather than the end and re-measure; a model with a weak position sense should
not care. (c) Using `reader.loss_table`, plot the validation curve and mark the step at which the loss passes
2.0 nats; then estimate, from the shape, how many steps a gap of 30 per cent would need — and say why that
estimate is not a forecast.

In [ ]:
# Starting point for exercise 3: the years the questions name, and whether the corpus has them.
YEAR = re.compile(r"\b(19|20)\d{2}\b")
for q in gold:
    if q.answerable:
        continue
    match = YEAR.search(q.question)
    named = match.group(0) if match else None
    in_corpus = named is not None and any(s.date[:4] == named for s in statements)
    print(f"{q.id}  year named: {str(named):<6}  a statement from that year exists: {in_corpus}")
print("\nTwo of the six name a year the corpus does not hold. Restricting the candidate chunks by that year")
print("before scoring is a hard constraint; the threshold is a soft one. Build the hard one and re-grade.")

---
**What the gates were.** **A**: seven turns, the whole ledger, the verifier's five checks; the token
accounting of the re-send; two calls stopped by two different mechanisms and a budget that halts mid-task.
**B**: Deck 6.2's toy built here, or read from the deck's run where no encoder is installed, and checked against
that run (1,264 passages at the deck's addresses, four files hash-checked, the slide's five rows, the prompt's
length); its Step 5 on the 26 graded questions (8 and 20 of 20 answerable, 4 of 6 unanswerable emptied by the
date filter) and `toy_rag.py` returning the same rows and prompt; 245 statements whose hashes were checked before a
byte was read, and 20 gold anchors proved to name statements the corpus holds; two chunkings whose counts and
mid-sentence cuts reproduce the recorded run; all six chunking-by-retriever combinations reproducing every
recall@k, every MRR and all 20 per-question ranks. **C**: the refusal threshold and all 26 evidence scores;
recall and MRR before and after the threshold; the two questions that slip through, and `1987` proved absent
from the fitted vocabulary; a stale index that cannot reach its anchor at any rank; a boundary that cuts a
173-character sentence after 87 characters; an intended meeting at rank 11 undated and rank 1 dated; the
reader's four file hashes, its parameter count and its four recorded losses. **D**: every quoted sentence
found verbatim in a retrieved chunk, carrying its chunk id and date. **E**: four probes against two settings
files; 44 attempted runs across three recorded scenarios and three re-checked transcripts.

None of them is a threshold on a result. **Gate B4 compares all six mean reciprocal ranks against the
recorded run, fusion's included; it would pass unchanged if fusion had beaten BM25. Gate C6 compares the
reader's four losses against the recorded run; it would pass unchanged at a gap of zero.** Nothing in this
notebook passes because a number came out in a direction the lab would like.

In [ ]:
print(f"{len(GATES)} gates, {sum(ok for _, ok in GATES)} passing:")
print("   " + "  ".join(name for name, _ in GATES))
print(f"notebook run time: {time.time() - T_START:.0f} s")